# Lab 10: How to choose between long context and retrieval

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Self-study**

## Goal

Decide between putting the whole corpus in the prompt and retrieving a few chunks, from answers, tokens, cost and
seconds measured on the same questions. The moment that teaches it: the whole corpus fits in one prompt, about
19,466 tokens, yet on `gpt-4o-mini` it passed the answer check for 25 of 34 questions against
31 for retrieval, at 40 times the input tokens per question. Given the supporting passage alone, the
model answered the questions it had missed in 18 of 18 runs; inside the whole corpus, in 0 of 18.

## What you will learn

- How to answer the same questions three ways (no evidence, three retrieved chunks, the whole corpus) and record the
  tokens and seconds of every answer.
- How to compare the three routes with one answer check, and read every answer it flags.
- How to price each route per 1,000 questions, and what a cached prefix changes (an OpenAI-only feature).
- How an answer changes when the passage that holds it gets more text around it, or moves.
- How to write a decision rule from your own table, and see where the corpus stops fitting in the prompt.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-18
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, `TOKEN_ENCODING`, `JSON_MODE` and the three
`CHAT_PRICE_*` lines from `models.env`; update the prices when you change the model.

⚠️ This lab costs more than the others: more than 150 of its chat calls send the whole corpus, about 20,000 tokens
each. Our run used 2,836,809 input tokens in all, about 0.31 US dollars on `gpt-4o-mini` (the last cell
of Part E prints yours). On a
rate-limit error, wait a minute and run the cell again. The saved outputs and every count in the text come from
`gpt-4o-mini`; the README lists what `gpt-6-luna` did.

---

## Part A: Answer one question three ways

Part A builds the three routes and runs them on two questions.

| Route | What the model reads besides the question | What it needs first |
|---|---|---|
| No evidence | nothing: it answers from what it learned in training | nothing |
| Retrieval | the three chunks closest to the question, Lab 05's pipeline | an index of the 277 chunks |
| Whole corpus | every passage and policy, as `[ID] text` lines | nothing: no chunks, no index |

The whole corpus is the 120 SQuAD paragraphs and the five policies, each as one `[ID] text` evidence line, so the
model can cite it and Lab 03's quote check can read it. Count the tokens:

In [3]:
import json
import tiktoken

DATA = ROOT / "data"
passages = json.loads((DATA / "squad-passages.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus_lines = [f"[{doc['source_id']}] {doc['text']}" for doc in passages + policies]

encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])
corpus_tokens = len(encoding.encode("\n".join(corpus_lines)))
print(len(corpus_lines), "lines |", corpus_tokens, "tokens")
print(corpus_lines[0][:100], "...")

125 lines | 19466 tokens
[S00-P00] Super Bowl 50 was an American football game to determine the champion of the National Foot ...


19,466 tokens: the paragraphs' text, plus an ID on each line and the policies. `gpt-4o-mini` reads up to
128,000 tokens in one request, so the whole corpus fits several times over. Part E comes back to that limit.

The retrieval route is Lab 05's pipeline from `shared/lc_pipeline.py`: the 277 chunks in an `InMemoryVectorStore`, and
`retrieve()` with `k=3`. Build the store and retrieve for the Tesla question:

In [4]:
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
question = "In what year was Nikola Tesla born?"
for line in lc_pipeline.evidence(lc_pipeline.retrieve(store, question)):
    print(len(encoding.encode(line)), "tokens |", line[:80], "...")

73 tokens | [S03-P00-C00] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 Jan ...
102 tokens | [S03-P01-C00] Tesla gained experience in telephony and electrical engineering be ...
100 tokens | [S03-P02-C00] Tesla went on to pursue his ideas of wireless lighting and electri ...


Three chunks, 275 tokens of evidence, where the whole-corpus route sends 19,466. Both
routes answer with Lab 05's `answer()`: the same rules and the same `CitedAnswer` schema, so only the evidence differs.
Read it:

In [5]:
import inspect

print(inspect.getsource(lc_pipeline.answer))

def answer(question: str, lines: list[str], settings: dict, rules: str = RULES) -> CitedAnswer:
    """One chat call through lc_chat_model() that returns a CitedAnswer (JSON_MODE picks a strict schema or JSON)."""
    method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
    model = lc_chat_model("CHAT_MODEL", settings).with_structured_output(CitedAnswer, method=method)
    user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
    return model.invoke([("system", rules), ("user", user)])



`answer()` returns only the `CitedAnswer`. To price a route you also need the tokens billed. With
`include_raw=True`, `with_structured_output` returns a dictionary instead
([LangChain docs](https://docs.langchain.com/oss/python/langchain/models#structured-output)): `parsed`, the
`CitedAnswer`, and `raw`, the model's `AIMessage` with `usage_metadata`. `ask()` below is `answer()` with that one
change, and it keeps the evidence lines so the quotes can be checked later. A **callback**, an object LangChain calls
after every model call, adds up the tokens of every call in the notebook:

In [6]:
from langchain_core.callbacks import UsageMetadataCallbackHandler
from shared.answer import RULES
from shared.settings import lc_chat_model

usage_total = UsageMetadataCallbackHandler()     # adds up the tokens of every call; Part E prints the total
chat = lc_chat_model("CHAT_MODEL", settings)
chat.callbacks = [usage_total]
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
structured = chat.with_structured_output(lc_pipeline.CitedAnswer, method=method, include_raw=True)

def ask(question, lines):
    """lc_pipeline.answer() with include_raw=True: the answer, its claims, the evidence and the tokens billed."""
    user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
    out = structured.invoke([("system", RULES), ("user", user)])
    return {"answer": out["parsed"].answer, "claims": [claim.model_dump() for claim in out["parsed"].claims],
            "lines": lines, "usage": out["raw"].usage_metadata}

record = ask(question, lc_pipeline.evidence(lc_pipeline.retrieve(store, question)))
print(record["answer"])
print(record["usage"])

Nikola Tesla was born in the year 1856.
{'input_tokens': 506, 'output_tokens': 51, 'total_tokens': 557, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


Now the three routes, as functions of the question. The no-evidence route gets a one-line instruction, since there is
no evidence to cite. `run()` times a route from the question to the answer, the retrieval route's search included.

In [7]:
import time

def no_evidence(question):
    reply = chat.invoke([("system", "Answer the question in one sentence."), ("user", question)])
    return {"answer": reply.content, "claims": [], "lines": [], "usage": reply.usage_metadata}

def retrieval(question):
    return ask(question, lc_pipeline.evidence(lc_pipeline.retrieve(store, question)))

def whole_corpus(question):
    return ask(question, corpus_lines)

def run(route, question):
    start = time.perf_counter()
    record = route(question)
    record["seconds"] = time.perf_counter() - start
    return record

ROUTES = {"no evidence": no_evidence, "retrieval": retrieval, "whole corpus": whole_corpus}

Ask the Tesla question and one about Atlas, a product made up for these labs, so no model learned its policies in
training. Predict first: which route will get the Atlas question wrong, and how?

In [8]:
for q in (question, "How long does an Atlas reset link remain valid?"):
    print(q)
    for name, route in ROUTES.items():
        record = run(route, q)
        print(f"  {name:13} {record['usage']['input_tokens']:6} tokens in | {record['seconds']:.2f} s | {record['answer']}")

In what year was Nikola Tesla born?


  no evidence       26 tokens in | 0.67 s | Nikola Tesla was born in the year 1856.


  retrieval        506 tokens in | 3.10 s | Nikola Tesla was born in the year 1856.


  whole corpus   19696 tokens in | 1.46 s | Nikola Tesla was born on 10 July 1856.
How long does an Atlas reset link remain valid?


  no evidence       28 tokens in | 0.99 s | An Atlas reset link typically remains valid for 24 hours.


  retrieval        296 tokens in | 1.32 s | Atlas reset links expire after 30 minutes.


  whole corpus   19698 tokens in | 2.37 s | Atlas reset links expire after 30 minutes.


All three routes give 1856 for Tesla, a famous fact the model learned in training. For Atlas, the no-evidence
route answers *An Atlas reset link typically remains valid for 24 hours.*: fluent and invented, since the policy says
30 minutes, and nothing in the reply says it is a guess. Retrieval and the whole corpus both say 30 minutes. Input tokens: 26 or 28 (no evidence), 506 or 296 (retrieval), about 19,700 (whole corpus). The seconds are single calls (0.67 to 3.10); Part C reads them over 34 questions.

### Checkpoint A

| You should see | Our run |
|---|---|
| The corpus | 125 lines, 19,466 tokens |
| Tesla, retrieved | three chunks from the Tesla article, 275 tokens |
| Tesla, three routes | 1856 from every route; 26, 506 and 19,696 input tokens |
| Atlas, three routes | *24 hours* with no evidence; *30 minutes* from retrieval and from the whole corpus |

---

## Part B: Answer every question three ways

Part B runs the three routes on Lab 04's questions: the 30 development questions with SQuAD's answer strings, and
four about Atlas with the answers the policies give.

In [9]:
dev = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
questions = [{"question": q["question"], "expected": sorted({a["text"] for a in q["answers"]}),
              "support": q["known_support"]} for q in dev]
questions += [  # written for Lab 04: the answers the Atlas policies give
    {"question": "How long does an Atlas reset link remain valid?", "expected": ["30 minutes"], "support": ["H02"]},
    {"question": "Who can export all reports in Atlas?", "expected": ["workspace owners"], "support": ["H03"]},
    {"question": "Can temporary Atlas accounts reset by email?", "expected": ["support agent"], "support": ["H01"]},
    {"question": "Do I need the mobile app to reset my Atlas password?", "expected": ["no", "by email"], "support": ["H01"]},
]
print(len(questions), "questions")

34 questions


One answer per question and route: 102 calls.

In [10]:
results = {name: [run(route, q["question"]) for q in questions] for name, route in ROUTES.items()}
print({name: len(records) for name, records in results.items()})

{'no evidence': 34, 'retrieval': 34, 'whole corpus': 34}


Check every answer with Lab 04's **screen**: it passes when one of the expected strings appears in the answer, as
whole words in lower case. A screen flags answers to read; it does not grade them.

In [11]:
from shared.answer import quote_found, words

INSUFFICIENT = "Insufficient evidence."

def screen(answer_text, expected):
    answer = f" {words(answer_text)} "
    return any(f" {words(text)} " in answer for text in expected)

for name, records in results.items():
    passed = sum(screen(r["answer"], q["expected"]) for r, q in zip(records, questions))
    refused = sum(r["answer"].strip() == INSUFFICIENT for r in records)
    print(f"{name:13} passed {passed} of {len(questions)} | refused {refused}")

no evidence   passed 19 of 34 | refused 0
retrieval     passed 31 of 34 | refused 1
whole corpus  passed 25 of 34 | refused 4


Retrieval passes 31 of 34, the whole corpus 25 and the no-evidence route 19. The whole corpus refused 4 questions, retrieval 1. A count is not a verdict: print every answer the screen flags, route by route, and read them.

In [12]:
for name, records in results.items():
    print(f"\n{name}")
    for r, q in zip(records, questions):
        if not screen(r["answer"], q["expected"]):
            print(f"  {q['question'][:60]:60} | expected {q['expected'][0][:25]!r} | {r['answer'][:90]}")


no evidence
  What is a popular strolling destination for the Varsovians?  | expected 'Krasiński Palace Garden' | A popular strolling destination for Varsovians is the Royal Łazienki Park.
  How far from Warsaw does the Vistula river's environment cha | expected '15 kilometres' | The environment of the Vistula River changes noticeably within about 50 to 100 kilometers 
  What animals does the Vistula river's ecosystem include?     | expected 'otter, beaver and hundred' | The Vistula River's ecosystem includes a variety of animals such as fish (like the Europea
  When were the Normans in Normandy?                           | expected '10th and 11th centuries' | The Normans were in Normandy from the 10th century, particularly after the Treaty of Saint
  Who ruled the duchy of Normandy                              | expected 'Richard I' | The duchy of Normandy was ruled by a series of dukes, starting with Rollo in the 10th cent
  What is the original meaning of the word Norman?          

- The no-evidence route has 15 flagged: less famous facts missed from memory (*the Royal Łazienki Park*), and an
  invented policy for three of the four Atlas questions (*24 hours*, *administrators*, *Yes, ... by email*).
- Retrieval has the three answers Lab 04 traced: the complexity question names the field, the temporary-accounts
  question is refused, and the mobile-app question gets *Yes, password reset uses a mobile app.*, the Boreal policy.
- The whole corpus has 9: the same three (the temporary-accounts answer is now *Yes, ... can reset by email.*, the
  opposite of H01), four refusals of questions whose paragraph is in the prompt, the method question answered with
  the field's name, and *Norseman*, right in other words than the label's *Norseman, Viking* (Lab 04's incomplete
  label).

The retrieval route and the whole-corpus route read the same rules, and the whole corpus holds every chunk retrieval
found. More evidence gave fewer answers.

Last, the citations, checked with Lab 03's quote check (the no-evidence route has nothing to cite):

In [13]:
for name, records in results.items():
    claims = [(r, claim) for r in records for claim in r["claims"]]
    found = sum(quote_found(claim, r["lines"]) for r, claim in claims)
    print(f"{name:13} claims {len(claims)} | quotes found {found}")

no evidence   claims 0 | quotes found 0
retrieval     claims 35 | quotes found 35
whole corpus  claims 31 | quotes found 29


Retrieval made 35 claims, and all 35 quotes were found in the lines they cite; the whole corpus made 31, with 29 found, each citing a whole paragraph instead of a chunk. Long context does not lose the citations: written as `[ID] text` lines, the corpus can be cited and
checked like retrieved chunks. What it changed was the answers.

### Checkpoint B

| You should see | Our run |
|---|---|
| The screen | no evidence 19, retrieval 31, whole corpus 25 of 34 |
| Refused | no evidence 0, retrieval 1, whole corpus 4 |
| Citations | no evidence none; retrieval 35 claims, 35 quotes found; whole corpus 31 claims, 29 found |

---

## Part C: Price each route

Part C puts the cost of each route beside its answers: tokens and dollars per question, and seconds.

The prices come from `models.env`, in US dollars per million tokens. The screen count stays in the first column: a
fast failure is always the cheapest row.

In [14]:
from shared.settings import chat_url

PRICE_IN, PRICE_OUT = float(settings["CHAT_PRICE_INPUT"]), float(settings["CHAT_PRICE_OUTPUT"])

def per_question(records):
    tokens_in = sum(r["usage"]["input_tokens"] for r in records) / len(records)
    tokens_out = sum(r["usage"]["output_tokens"] for r in records) / len(records)
    return tokens_in, tokens_out, (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1e6

print(f"{'route':13} {'passed':>6} {'in/question':>12} {'out/question':>13} {'$ per 1,000':>12}  seconds: median (range)")
for name, records in results.items():
    passed = sum(screen(r["answer"], q["expected"]) for r, q in zip(records, questions))
    tokens_in, tokens_out, dollars = per_question(records)
    seconds = sorted(r["seconds"] for r in records)
    print(f"{name:13} {passed:6} {tokens_in:12,.0f} {tokens_out:13,.0f} {1000 * dollars:12.2f}  "
          f"{seconds[len(seconds) // 2]:.2f} ({seconds[0]:.2f} to {seconds[-1]:.2f})")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}; prices from models.env")

route         passed  in/question  out/question  $ per 1,000  seconds: median (range)
no evidence       19           30            20         0.02  0.72 (0.56 to 1.10)
retrieval         31          489            70         0.12  1.43 (0.93 to 1.74)
whole corpus      25       19,700            68         3.00  1.63 (1.27 to 3.47)

gpt-4o-mini at https://api.openai.com/v1; prices from models.env


Retrieval passes 31 for 0.12 dollars per 1,000 questions; the whole corpus passes 25 for 3.00. Its input is 40
times retrieval's (19,700 against 489 tokens a question), and its price 25 times, because both routes write about the
same answer. The seconds differ least: medians of 1.43 and 1.63 with overlapping ranges, from one run. The
no-evidence route is the cheapest and fastest row (0.02 dollars, 0.72 seconds), and the one that invents policies.

Retrieval also pays for its index: the 277 chunks embedded once, well under a cent, and one short embedding per
question. The whole corpus needs no index, and pays for the corpus again on every question.

### A cached prefix (OpenAI only)

OpenAI caches the beginning of recent prompts. When a new request starts with the same tokens as a recent one, for
at least a minimum length (about 1,024 tokens; the guide gives it per model), those tokens are read from the cache
and billed at the cached price, `CHAT_PRICE_CACHED_INPUT`, half the input price for `gpt-4o-mini`
([prompt caching guide](https://developers.openai.com/api/docs/guides/prompt-caching)). LangChain reports
them as `cache_read` in `usage_metadata["input_token_details"]`; an endpoint that does not report them prints 0. How
much of Part B's whole-corpus input was cached?

In [15]:
def cached(record):
    return record["usage"].get("input_token_details", {}).get("cache_read", 0)

whole = results["whole corpus"]
print(f"cached {sum(cached(r) for r in whole):,} of {sum(r['usage']['input_tokens'] for r in whole):,} input tokens")

cached 19,584 of 669,799 input tokens


19,584 of 669,799: one call's worth, a question Part A had already asked with the same prompt. (Run the notebook
again within about half an hour and much more is cached: every prompt repeats.) The question comes before the corpus, so two questions share only the rules. Put the corpus first,
and every question starts with the same 19,466 tokens:

In [16]:
def whole_corpus_first(question):
    user = "Evidence:\n" + "\n".join(corpus_lines) + f"\nQuestion: {question}"
    out = structured.invoke([("system", RULES), ("user", user)])
    return {"answer": out["parsed"].answer, "claims": [claim.model_dump() for claim in out["parsed"].claims],
            "lines": corpus_lines, "usage": out["raw"].usage_metadata}

first = [run(whole_corpus_first, q["question"]) for q in questions]
print(f"cached {sum(cached(r) for r in first):,} of {sum(r['usage']['input_tokens'] for r in first):,} input tokens")

cached 626,688 of 669,798 input tokens


Price it at the cached price, and screen its answers: a change made for cost is still a change to the prompt.

In [17]:
PRICE_CACHED = float(settings["CHAT_PRICE_CACHED_INPUT"])

def dollars(r):
    full = r["usage"]["input_tokens"] - cached(r)
    return (full * PRICE_IN + cached(r) * PRICE_CACHED + r["usage"]["output_tokens"] * PRICE_OUT) / 1e6

for name, records in [("question first", whole), ("corpus first", first)]:
    passed = sum(screen(r["answer"], q["expected"]) for r, q in zip(records, questions))
    refused = sum(r["answer"].strip() == INSUFFICIENT for r in records)
    print(f"{name:14} passed {passed} of 34 | refused {refused} | $ per 1,000 questions {1000 * sum(map(dollars, records)) / len(records):.2f}")
print("screen result changed:", [q["question"][:45] for q, a, b in zip(questions, whole, first)
                                 if screen(a["answer"], q["expected"]) != screen(b["answer"], q["expected"])])

question first passed 25 of 34 | refused 4 | $ per 1,000 questions 2.95
corpus first   passed 22 of 34 | refused 10 | $ per 1,000 questions 1.61
screen result changed: ['What was Maria Curie the first female recipie', 'What year was Casimir Pulaski born in Warsaw?', "What animals does the Vistula river's ecosyst", 'In what country is Normandy located?', 'Who ruled the duchy of Normandy', 'In what year was Nikola Tesla born? ', "What was Nikola Tesla's ethnicity?", 'In what year did Nikola Tesla emigrate to the', 'When did Tesla discover the possibility of wi', 'What measure of a computational problem broad', 'What method is used to intuitively assess or ', 'What field of computer science analyzes the r', 'Can temporary Atlas accounts reset by email?']


With the corpus first, 626,688 of 669,798 input tokens were cached, and the cost per 1,000 questions fell from
2.95 to 1.61 dollars. The answers moved too: 22 passed instead of 25, 10 were refused instead of 4, and 13 questions
changed their screen result. On `gpt-4o-mini` the order of the prompt is part of the answer. Retrieval's prompts, about 500
tokens, are too short to cache and cheap without it.

### Checkpoint C

| You should see | Our run |
|---|---|
| Per 1,000 questions | no evidence 0.02, retrieval 0.12, whole corpus 3.00 dollars |
| Input per question | retrieval 489, whole corpus 19,700 tokens (40 times) |
| Cached, question first | 19,584 of 669,799 tokens |
| Cached, corpus first | 626,688 of 669,798 tokens; 22 passed, 10 refused; 1.61 dollars per 1,000 |

---

## Part D: Give the passage more text, or move it

Part D asks why the whole corpus lost answers that retrieval found.

In [18]:
lost = [q for q, r, w in zip(questions, results["retrieval"], results["whole corpus"])
        if screen(r["answer"], q["expected"]) and not screen(w["answer"], q["expected"])]
for q in lost:
    print(q["support"], "|", q["question"])

['S02-P01'] | Who was the duke in the battle of Hastings?
['S02-P01'] | Who ruled the duchy of Normandy
['S02-P02'] | What is the original meaning of the word Norman?
['S02-P02'] | When was the Latin version of the word Norman first recorded?
['S04-P01'] | What method is used to intuitively assess or quantify the amount of resources required to solve a computational problem?
['S04-P02'] | What field of computer science analyzes the resource requirements of a specific algorithm isolated unto itself within a given problem?


Six questions: four about the Normans (`S02-P01`, `S02-P02`) and two about complexity theory (`S04-P01`,
`S04-P02`). One is not a real loss: the whole corpus answered *Norseman*, right in other words. For each one, build six contexts around its supporting passage, the paragraph its label names. The
first three add text around it; the last three keep the whole corpus and move the passage.

| Context | What the model reads |
|---|---|
| passage alone | the supporting passage only, Lab 04's oracle context |
| its article | the three paragraphs of the same article |
| whole corpus | all 125 lines, as in Part B |
| moved first, to the middle, last | all 125 lines, with the passage taken out of its place and put first, 63rd or last |

In [19]:
line_of = {line[1:line.index("]")]: line for line in corpus_lines}

def contexts(source_id):
    support = line_of[source_id]
    rest = [line for line in corpus_lines if line != support]
    return {"passage alone": [support],
            "its article": [line for line in corpus_lines if line.startswith(f"[{source_id[:3]}")],
            "whole corpus": corpus_lines,
            "moved first": [support] + rest,
            "moved to middle": rest[:62] + [support] + rest[62:],
            "moved last": rest + [support]}

for name, lines in contexts(lost[0]["support"][0]).items():
    print(f"{name:16} {len(lines):3} lines, passage at line {lines.index(line_of[lost[0]['support'][0]]) + 1}")

passage alone      1 lines, passage at line 1
its article        3 lines, passage at line 2
whole corpus     125 lines, passage at line 8
moved first      125 lines, passage at line 1
moved to middle  125 lines, passage at line 63
moved last       125 lines, passage at line 125


Ask each question three times in each context (108 calls, 72 with the whole corpus). Predict
first: which contexts will answer?

In [20]:
from collections import Counter

answers = {}
for q in lost:
    for name, lines in contexts(q["support"][0]).items():
        answers[q["question"], name] = [ask(q["question"], lines)["answer"] for _ in range(3)]

for name in contexts(lost[0]["support"][0]):
    row = [sum(screen(a, q["expected"]) for a in answers[q["question"], name]) for q in lost]
    print(f"{name:16} passed {sum(row):2} of {3 * len(lost)} | per question {row}")

passage alone    passed 18 of 18 | per question [3, 3, 3, 3, 3, 3]
its article      passed 15 of 18 | per question [3, 3, 0, 3, 3, 3]
whole corpus     passed  0 of 18 | per question [0, 0, 0, 0, 0, 0]
moved first      passed 18 of 18 | per question [3, 3, 3, 3, 3, 3]
moved to middle  passed  6 of 18 | per question [0, 3, 0, 3, 0, 0]
moved last       passed 12 of 18 | per question [3, 3, 3, 3, 0, 0]


Read the answers, not only the counts:

In [21]:
for q in lost:
    print(f"\n{q['question']}  (expected {q['expected'][0]!r})")
    for name in contexts(q["support"][0]):
        replies = answers[q["question"], name]
        passed = sum(screen(a, q["expected"]) for a in replies)
        print(f"  {name:16} passed {passed} of 3 | {Counter(replies).most_common(1)[0][0][:85]}")


Who was the duke in the battle of Hastings?  (expected 'William the Conqueror')
  passage alone    passed 3 of 3 | The duke in the Battle of Hastings was William the Conqueror.
  its article      passed 3 of 3 | The duke in the Battle of Hastings was William the Conqueror.
  whole corpus     passed 0 of 3 | Insufficient evidence.
  moved first      passed 3 of 3 | The duke in the Battle of Hastings was William the Conqueror.
  moved to middle  passed 0 of 3 | Insufficient evidence.
  moved last       passed 3 of 3 | The duke in the Battle of Hastings was William the Conqueror.

Who ruled the duchy of Normandy  (expected 'Richard I')
  passage alone    passed 3 of 3 | The duchy of Normandy was ruled by Richard I of Normandy.
  its article      passed 3 of 3 | The duchy of Normandy was ruled by Richard I of Normandy.
  whole corpus     passed 0 of 3 | Insufficient evidence.
  moved first      passed 3 of 3 | The duchy of Normandy was ruled under Richard I of Normandy, who forged it into

Alone, the passage answers in 18 of 18 runs; inside its article in 15 of 18 (the misses are *Norseman or Northman*,
the label again); inside the whole corpus in 0 of 18, mostly *Insufficient evidence.* Only the text around it changed.

Moving the passage changed the answers again: first in the prompt 18 of 18, last 12 of 18, in the middle 6 of 18. Its
own place, line 8, near the start, gave 0, so neither "near the start is safe" nor "the middle is lost" holds. Moved
to the middle, both complexity questions name the field (*Computational complexity theory ...*). Whether an answer
survives a long prompt depends on where its passage sits and what surrounds it, and only a run on your questions
shows it. Retrieval's prompt holds three chunks, the closest one first.

### Checkpoint D

| You should see | Our run |
|---|---|
| Questions lost | six: four about the Normans, two about complexity theory |
| More text around the passage | alone 18 of 18, its article 15 of 18, whole corpus 0 of 18 |
| The passage moved | first 18 of 18, to the middle 6 of 18, last 12 of 18 |

---

## Part E: Write the decision rule

Part E turns the measurements into a rule. The whole-corpus route's tokens grow with the corpus; retrieval sends
three chunks at any size. From Part B's averages, the cost as the corpus grows:

In [22]:
ret_in, ret_out, _ = per_question(results["retrieval"])
whole_in, whole_out, _ = per_question(results["whole corpus"])
overhead = whole_in - corpus_tokens            # the rules, the question and the schema

print(f"{'corpus size':>12} {'whole corpus: in/question':>26} {'$ per 1,000':>12} | retrieval: $ per 1,000")
for times in (1, 5, 10, 100):
    tokens = times * corpus_tokens + overhead
    print(f"{times:>11}x {tokens:26,.0f} {(tokens * PRICE_IN + whole_out * PRICE_OUT) / 1e3:12.2f} |"
          f" {(ret_in * PRICE_IN + ret_out * PRICE_OUT) / 1e3:.2f}")

 corpus size  whole corpus: in/question  $ per 1,000 | retrieval: $ per 1,000
          1x                     19,700         3.00 | 0.12
          5x                     97,564        14.68 | 0.12
         10x                    194,894        29.27 | 0.12
        100x                  1,946,834       292.07 | 0.12


The model has a limit on what one request can hold, its **context window**. Stand in for a corpus seven times larger
by repeating the lines seven times, and ask the Tesla question. `ContextOverflowError` is LangChain's error for a
prompt longer than the window, whatever the provider; the cell prints it instead of stopping:

In [23]:
from langchain_core.exceptions import ContextOverflowError

bigger = corpus_lines * 7
print(len(encoding.encode("\n".join(bigger))), "tokens")
try:
    print(ask(question, bigger)["answer"])
except ContextOverflowError as error:
    print(type(error).__name__, "|", error)

136262 tokens


OpenAIContextOverflowError | Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 136492 tokens (including 107 in the response_format schemas.). Please reduce the length of the messages or schemas.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


136,262 tokens by the local count, and the API refused the request: *This model's maximum context length is
128000 tokens.* Five copies of this corpus still fit (97,564 tokens a question in the table above); seven do not. Long
before that, the price grows: 14.68 dollars per 1,000 questions at five times this corpus, against 0.12. A larger
window moves the limit; it does not lower the cost.

Put the measurements together. The rule below is written from this run on `gpt-4o-mini`; write yours from your own
table, on your model:

| If | Choose | What this run measured |
|---|---|---|
| The answer is private or recent, or must name its source | not the no-evidence route | 19 of 34 passed; three of the four Atlas answers invented; nothing to cite |
| The corpus fits in the window, questions are few, and you have measured the answers on your model | the whole corpus can be an option; put it before the question so it is cached | 25 of 34 at 3.00 dollars per 1,000 (1.61 cached, 22 passing); lost questions 0 of 18 runs |
| The corpus is large or growing, questions are many, or some documents must be left out for some users | retrieval | 31 of 34 at 0.12 dollars per 1,000, three chunks a question at any corpus size |

A window size decides neither of the two things that matter most: how well your model answers from a long prompt,
and whether some documents must be left out, which takes a search with a filter (Lab 05, Part G).

Last, what the whole notebook cost, from the callback added in Part A:

In [24]:
for model_name, used in usage_total.usage_metadata.items():
    cached_in = used.get("input_token_details", {}).get("cache_read", 0)
    cost = ((used["input_tokens"] - cached_in) * PRICE_IN + cached_in * PRICE_CACHED + used["output_tokens"] * PRICE_OUT) / 1e6
    print(f"{model_name}: {used['input_tokens']:,} input tokens ({cached_in:,} cached), {used['output_tokens']:,} output | {cost:.2f} dollars")

gpt-4o-mini-2024-07-18: 2,836,809 input tokens (1,684,736 cached), 17,739 output | 0.31 dollars


2,836,809 input tokens, 1,684,736 of them cached, and 0.31 dollars for the whole notebook, most of it the
whole-corpus calls. Each table above priced only the calls it showed; the callback counted every one.

### Checkpoint E

| You should see | Our run |
|---|---|
| Corpus growth | whole corpus 19,700 tokens and 3.00 dollars per 1,000 at 1x, 97,564 tokens and 14.68 at 5x; retrieval 0.12 at every size |
| Seven copies | 136,262 tokens; `OpenAIContextOverflowError`, maximum 128,000 tokens |
| The notebook's total | 2,836,809 input tokens (1,684,736 cached), 0.31 dollars |

---

## What you can do now

- Answer the same questions with no evidence, with retrieved chunks and with the whole corpus, and record the tokens
  billed and the seconds of every answer.
- Compare routes with one answer check, and read every flagged answer before you trust a count.
- Price each route per 1,000 questions from `models.env`, and say what a cached prefix saves and what it changed.
- Show how an answer changes when its passage gets more text around it or moves, with a count over several runs.
- Write a decision rule from your own table, and name the corpus size at which the whole corpus no longer fits.

## Going further: is the abstention line the cause?

Most lost answers were *Insufficient evidence.*, the abstention line's reply. Take the line out, as Lab 04 did, and
ask the lost questions three times each with the whole corpus, plus Lab 03's Moon question, which the corpus cannot
answer:

In [25]:
ABSTAIN = ' If the evidence does not answer the question, the answer is "Insufficient evidence." and the claims list is empty.'
rules_without = RULES.replace(ABSTAIN, "")
print("abstention line removed:", rules_without != RULES)

for q in lost + [{"question": "Who was the first person to walk on the Moon?", "expected": ["Neil Armstrong"]}]:
    user = f"Question: {q['question']}\nEvidence:\n" + "\n".join(corpus_lines)
    replies = [structured.invoke([("system", rules_without), ("user", user)])["parsed"].answer for _ in range(3)]
    print(f"passed {sum(screen(a, q['expected']) for a in replies)} of 3 | {q['question'][:50]:50} | {replies[0][:70]}")

abstention line removed: True


passed 3 of 3 | Who was the duke in the battle of Hastings?        | The duke in the Battle of Hastings was William the Conqueror.


passed 2 of 3 | Who ruled the duchy of Normandy                    | The duchy of Normandy was ruled by the Normans, who were led by Rollo 


passed 0 of 3 | What is the original meaning of the word Norman?   | The original meaning of the word Norman is "Norseman," derived from th


passed 3 of 3 | When was the Latin version of the word Norman firs | The Latin version of the word Norman was first recorded in the 9th cen


passed 0 of 3 | What method is used to intuitively assess or quant | The method used to intuitively assess or quantify the amount of resour


passed 3 of 3 | What field of computer science analyzes the resour | The field of computer science that analyzes the resource requirements 


passed 0 of 3 | Who was the first person to walk on the Moon?      | The evidence does not provide information about who was the first pers


Without the line, the whole corpus passes 11 of these 18 runs: the duke at Hastings, the Latin word and the
field that analyses one algorithm 3 of 3 each, the duchy 2 of 3. The other two stay wrong: *Norseman*, the label again,
and the method question, which still names the field. The Moon question is still declined: no run names Neil
Armstrong, and the first reply says *The evidence does not provide information about who was the first person ...*.
Most lost answers came from the abstention line meeting a long prompt. Lab 03 measured it on a few retrieved chunks;
a rule measured on one route has to be measured again on another.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Move a passage the model already answers from.** Part D moved the passages of lost questions. Does moving a
   passage also break a question the whole corpus answered?

   - Under **Task 1**, `kept` holds five questions both routes passed in Part B. Complete the loop: ask each three
     times with its passage moved to the middle (`contexts(...)["moved to middle"]`) and count the passes.
   - Run the cell.

   **Check:** five lines with a count of 3 each at most, and a total. Our worked solution got 15 of 15: moving the passage to the middle broke none of the five.

   **Explain:** what does your count, beside Part D's, say about a rule such as "put the evidence at the ends of the
   prompt"?

2. **Optional: price a workload.** A support team expects `QUESTIONS_PER_MONTH` questions on a corpus that will grow
   to `CORPUS_TIMES` times this one. Complete `monthly()` under **Task 2** with Part C's prices and Part B's
   per-question tokens, for the retrieval route and for the whole corpus with every corpus token cached.

   **Check:** two monthly costs in dollars, and whether the whole corpus still fits in 128,000 tokens. Our worked solution, with 50,000 questions a month and a corpus five times this one: 5.68 dollars a month for
retrieval, 368.65 for the whole corpus with the corpus cached, at 97,564 tokens a question, which still fits.

   **Explain:** at what number of questions per month would the difference stop mattering to you, and what would
   still decide the route?

**Task 1.** Complete the loop, then run the cell:

In [26]:
kept = [q for q, r, w in zip(questions, results["retrieval"], results["whole corpus"])
        if screen(r["answer"], q["expected"]) and screen(w["answer"], q["expected"])][::5][:5]

total = None
for q in kept:
    lines = contexts(q["support"][0])["moved to middle"]
    # TODO: ask the question three times with these lines (ask(q["question"], lines)), screen each answer,
    #       print the count with the question, and add it to total (start total at 0)
print(f"total: {total} of {3 * len(kept)}" if total is not None else "⚠️ Complete the loop, then run the cell again.")

⚠️ Complete the loop, then run the cell again.


**Task 2 (optional).** Complete `monthly()`, then run the cell:

In [27]:
QUESTIONS_PER_MONTH = 50_000
CORPUS_TIMES = 5

def monthly(tokens_in, tokens_out, cached_in=0):
    """Dollars per month for QUESTIONS_PER_MONTH questions with these tokens per question."""
    # TODO: price one question (the cached tokens at PRICE_CACHED, the other input tokens at PRICE_IN, the output
    #       at PRICE_OUT, all per million) and multiply by QUESTIONS_PER_MONTH
    return None

whole_tokens = CORPUS_TIMES * corpus_tokens + overhead
if monthly(1, 1) is None:
    print("⚠️ Complete monthly(), then run the cell again.")
else:
    print(f"retrieval:    {monthly(ret_in, ret_out):10,.2f} dollars per month")
    print(f"whole corpus: {monthly(whole_tokens, whole_out, CORPUS_TIMES * corpus_tokens):10,.2f} dollars per month,"
          f" {whole_tokens:,.0f} tokens per question, fits in 128,000: {whole_tokens <= 128_000}")

⚠️ Complete monthly(), then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `OpenAIContextOverflowError: Error code: 400 ... This model's maximum context length is 128000 tokens. However, your messages resulted in 136492 tokens ...` | the prompt is longer than the model's context window | Send less: retrieve, or split the corpus; a model with a larger window only moves the limit |
| `AttributeError: 'dict' object has no attribute 'answer'` | the reply of `with_structured_output(..., include_raw=True)` read like a `CitedAnswer` | Read `out["parsed"].answer`, as `ask()` does; the tokens are in `out["raw"].usage_metadata` |
| `ValueError: could not convert string to float: ''` in Part C | a price line in `models.env` or `.env` left empty | Fill in the price for your model, in US dollars per million tokens, or 0 if you do not pay per token |

## Resources

- [OpenAI: prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching): which prompts are cached,
  the minimum prefix length for each model, and `cached_tokens` in `usage`
- [OpenAI: pricing](https://developers.openai.com/api/docs/pricing): input, cached input and output prices per model,
  the source of the prices in `models.env`
- [LangChain: models, structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output):
  `include_raw=True` and the token counts in `usage_metadata`
- [Hong, Troynikov and Huber, *Context Rot*, Chroma, 2025](https://www.trychroma.com/research/context-rot): 18 models
  answering worse as the input grows, and how related text distracts them
- [Liu et al., *Lost in the Middle*, TACL 2023](https://arxiv.org/abs/2307.03172): how answers depended on where the
  relevant passage sat in a long prompt, on the models of 2023

Next: [Lab 11](../11_Release_Gate/README.md), *How to gate a RAG change before release*. Back to [the lab list](../README.md).